In [3]:
from pathlib import Path
import re
import html
import unicodedata
import pandas as pd


# RAW_XML_DIR = Path("data/dart/raw_xml")
# COMPANY_MASTER_PATH = Path("data/company_master.csv")

RAW_XML_DIR = Path("../data/dart/raw_xml")
COMPANY_MASTER_PATH = Path("../data/company_master.csv")

TARGET_TITLE_REGEX = {
    "II. 사업의 내용": r"^(II|Ⅱ)\.\s*사업의\s*내용",
    "IV. 이사의 경영진단 및 분석의견": r"^(IV|Ⅳ)\.\s*이사의\s*경영진단\s*및\s*분석의견",
    "VI. 이사회 등 회사의 기관에 관한 사항": r"^(VI|Ⅵ)\.\s*이사회\s*등\s*회사의\s*기관에\s*관한\s*사항",
}

TITLE_RE = re.compile(r"<TITLE\b[^>]*>(.*?)</TITLE>", flags=re.I | re.S)
MAIN_TITLE_RE = re.compile(
    r"^\s*(I|II|III|IV|V|VI|VII|VIII|IX|X|Ⅰ|Ⅱ|Ⅲ|Ⅳ|Ⅴ|Ⅵ|Ⅶ|Ⅷ|Ⅸ|Ⅹ)\."
)


def clean_xml_text(text: str) -> str:
    text = "" if text is None else str(text)
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_report_text(text: str) -> str:
    text = clean_xml_text(text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_xml_filename(path: Path) -> tuple[str, int, str]:
    match = re.match(r"(\d{6})_(\d{4})_(\d+)\.xml$", path.name)
    if not match:
        raise ValueError(f"Unexpected XML filename: {path.name}")

    stock_code, fiscal_year, rcept_no = match.groups()
    return stock_code, int(fiscal_year), rcept_no


def extract_target_sections(xml_text: str) -> list[dict]:
    titles = []

    for match in TITLE_RE.finditer(xml_text):
        title = clean_xml_text(match.group(1))
        if MAIN_TITLE_RE.match(title):
            titles.append((title, match.start()))

    sections = []

    for i, (title, start) in enumerate(titles):
        section_name = None

        for name, pattern in TARGET_TITLE_REGEX.items():
            if re.search(pattern, title):
                section_name = name
                break

        if section_name is None:
            continue

        end = titles[i + 1][1] if i + 1 < len(titles) else len(xml_text)
        section_raw = xml_text[start:end]

        sections.append({
            "section": section_name,
            "text": clean_xml_text(section_raw),
        })

    return sections


def load_company_names(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=["stock_code", "company_name"])

    company_df = pd.read_csv(path, dtype={"stock_code": "string"})
    company_df["stock_code"] = (
        company_df["stock_code"]
        .astype("string")
        .str.extract(r"(\d+)", expand=False)
        .str.zfill(6)
    )

    if "company_name" not in company_df.columns:
        return pd.DataFrame(columns=["stock_code", "company_name"])

    return (
        company_df[["stock_code", "company_name"]]
        .dropna()
        .drop_duplicates("stock_code")
    )


def build_firm_year_corpus(raw_xml_dir: Path) -> pd.DataFrame:
    rows = []

    xml_files = sorted(raw_xml_dir.glob("*.xml"))
    if not xml_files:
        raise FileNotFoundError(f"No XML files found in {raw_xml_dir}")

    for xml_path in xml_files:
        stock_code, fiscal_year, rcept_no = parse_xml_filename(xml_path)
        xml_text = xml_path.read_text(encoding="utf-8", errors="ignore")
        sections = extract_target_sections(xml_text)

        document = " ".join(section["text"] for section in sections)
        document_norm = normalize_report_text(document)

        rows.append({
            "stock_code": stock_code,
            "fiscal_year": fiscal_year,
            "rcept_no": rcept_no,
            "file_name": xml_path.name,
            "document": document,
            "document_norm": document_norm,
            "section_count": len({section["section"] for section in sections}),
            "total_word_count": len(document_norm.split()),
            "total_char_count": len(document_norm),
            "esg_year": fiscal_year + 1,
        })

    corpus_df = pd.DataFrame(rows)
    company_df = load_company_names(COMPANY_MASTER_PATH)

    corpus_df = corpus_df.merge(company_df, on="stock_code", how="left")

    ordered_cols = [
        "stock_code",
        "company_name",
        "fiscal_year",
        "rcept_no",
        "file_name",
        "document",
        "document_norm",
        "section_count",
        "total_word_count",
        "total_char_count",
        "esg_year",
    ]

    return (
        corpus_df[ordered_cols]
        .sort_values(["stock_code", "fiscal_year", "rcept_no"])
        .reset_index(drop=True)
    )


corpus_df = build_firm_year_corpus(RAW_XML_DIR)

print("rows:", len(corpus_df))
print("section_count distribution:")
print(corpus_df["section_count"].value_counts().sort_index())
corpus_df.head()

rows: 381
section_count distribution:
section_count
3    381
Name: count, dtype: int64


,stock_code,company_name,fiscal_year,rcept_no,file_name,document,document_norm,section_count,total_word_count,total_char_count,esg_year
0,000020,동화약품,2022,20230315001100,000020_2022_20230315001100.xml,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,3,7863,38149,2023
1,000020,동화약품,2023,20240319000652,000020_2023_20240319000652.xml,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,3,8120,38663,2024
2,000020,동화약품,2024,20250318000739,000020_2024_20250318000739.xml,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,3,8152,39455,2025
3,000040,KR모터스,2022,20230322001182,000040_2022_20230322001182.xml,II. 사업의 내용 1. 사업의 개요 가. 업계의 현황 수출주력시장인 유럽 불경기에...,II. 사업의 내용 1. 사업의 개요 가. 업계의 현황 수출주력시장인 유럽 불경기에...,3,4256,20332,2023
4,000040,KR모터스,2023,20240321002062,000040_2023_20240321002062.xml,II. 사업의 내용 1. 사업의 개요 가. 업계의 현황 수출주력시장인 유럽 불경기에...,II. 사업의 내용 1. 사업의 개요 가. 업계의 현황 수출주력시장인 유럽 불경기에...,3,4943,22598,2024
